# 02 — Leakage-safe dataset split

This notebook creates the reusable WM-811K data partition defined in `split_strategy.md`. The supervised observations are assigned to 70% train, 20% validation, and 10% test while keeping every `lotName` entirely within one subset. Unlabeled observations are retained separately and classified by their eligibility for future training-only SSL or pseudo-labeling.

## Method and safeguards

The procedure first builds ten approximately stratified folds at the lot level and then combines seven folds for train, two for validation, and one for test. This gives the desired 70/20/10 proportions without dividing a manufacturing lot. Candidate fold combinations are scored by their sample-size and per-class deviations from the targets.

The fixed seed is used only to resolve deterministic ordering ties. Validation and test remain untouched: sampling, loss weighting, and data augmentation belong exclusively to later training stages.

In [1]:
from itertools import combinations
from pathlib import Path
import json

import numpy as np
import pandas as pd

RANDOM_SEED = 86
TARGET_FRACTIONS = {"train": 0.70, "validation": 0.20, "test": 0.10}
SPLIT_NAMES = tuple(TARGET_FRACTIONS)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_PATH = PROJECT_ROOT / "data" / "MIR-WM811K" / "WM811K.pkl"
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset: {DATASET_PATH}")
print(f"Split artifacts: {SPLITS_DIR}")

Dataset: C:\Users\edoar\OneDrive\Desktop\UniMib_DataScience\FoundationsOfDeepLearning\fdl-project\data\MIR-WM811K\WM811K.pkl
Split artifacts: C:\Users\edoar\OneDrive\Desktop\UniMib_DataScience\FoundationsOfDeepLearning\fdl-project\data\splits


## Load only the required metadata

The pickle contains the wafer-map arrays as well as metadata. The complete object must be deserialized, but the split calculation retains only row indices, `lotName`, `failureType`, and the native `trainTestLabel`. Empty NumPy arrays or the numeric value zero represent missing labels.

In [2]:
def normalize_scalar_label(value):
    values = np.asarray(value, dtype=object).reshape(-1)
    if values.size == 0:
        return pd.NA
    value = values[0]
    if value is None or value == 0:
        return pd.NA
    text = str(value).strip()
    return text if text and text != "0" else pd.NA

raw = pd.read_pickle(DATASET_PATH)
metadata = pd.DataFrame(
    {
        "row_index": raw.index.to_numpy(dtype=np.int64),
        "lotName": raw["lotName"].astype("string"),
        "failureType": raw["failureType"].map(normalize_scalar_label).astype("string"),
        "nativeSplit": raw["trainTestLabel"].map(normalize_scalar_label).astype("string"),
    },
    index=raw.index,
)
del raw

labeled = metadata.loc[metadata["failureType"].notna()].copy()
unlabeled = metadata.loc[metadata["failureType"].isna()].copy()

print(f"Total: {len(metadata):,}")
print(f"Labeled: {len(labeled):,}")
print(f"Unlabeled: {len(unlabeled):,}")
print(f"Distinct lots: {metadata['lotName'].nunique():,}")

Total: 811,457
Labeled: 172,950
Unlabeled: 638,507
Distinct lots: 46,293


## Inspect the native split

The original field is retained for traceability but not reused. It has no validation partition, gives most labeled observations to test, and exhibits a large class-distribution shift. The comparison below also checks whether its training and test lots overlap.

In [3]:
native_counts = pd.crosstab(labeled["nativeSplit"], labeled["failureType"], margins=True)
display(native_counts)
display((labeled["nativeSplit"].value_counts(normalize=True) * 100).round(2).rename("percent"))

native_train_lots = set(labeled.loc[labeled["nativeSplit"] == "Training", "lotName"])
native_test_lots = set(labeled.loc[labeled["nativeSplit"] == "Test", "lotName"])
print(f"Native train/test lot overlap: {len(native_train_lots & native_test_lots):,}")

failureType,Center,Donut,Edge-Loc,Edge-Ring,Loc,Near-full,Random,Scratch,none,All
nativeSplit,,,,,,,,,,
Test,832,146,2772,1126,1973,95,257,693,110701,118595
Training,3462,409,2417,8554,1620,54,609,500,36730,54355
All,4294,555,5189,9680,3593,149,866,1193,147431,172950


nativeSplit
Test        68.57
Training    31.43
Name: percent, dtype: Float64

Native train/test lot overlap: 0


## Build ten approximately stratified group folds

Each row of `lot_class_counts` represents a complete lot and each column a supervised class. Lots with difficult or rare class compositions are assigned first. For every lot, the algorithm selects the fold that minimizes dispersion of per-class and total-sample proportions across the ten folds.

In [4]:
lot_class_counts = pd.crosstab(labeled["lotName"], labeled["failureType"]).sort_index()
class_names = lot_class_counts.columns.tolist()
group_counts = lot_class_counts.to_numpy(dtype=np.int64)
class_totals = group_counts.sum(axis=0)
group_sizes = group_counts.sum(axis=1)

def build_group_folds(counts, seed, n_folds=10):
    rng = np.random.default_rng(seed)
    totals = counts.sum(axis=0)
    sizes = counts.sum(axis=1)
    fold_counts = np.zeros((n_folds, counts.shape[1]), dtype=np.int64)
    fold_sizes = np.zeros(n_folds, dtype=np.int64)
    assignments = np.full(len(counts), -1, dtype=np.int8)

    difficulty = np.std(counts / np.maximum(totals, 1), axis=1)
    order = np.lexsort((rng.random(len(counts)), -sizes, -difficulty))

    for group_id in order:
        candidate_scores = []
        for fold_id in range(n_folds):
            fold_counts[fold_id] += counts[group_id]
            fold_sizes[fold_id] += sizes[group_id]
            class_score = np.mean(np.std(fold_counts / np.maximum(totals, 1), axis=0))
            size_score = np.std(fold_sizes / sizes.sum())
            candidate_scores.append(class_score + 0.20 * size_score)
            fold_counts[fold_id] -= counts[group_id]
            fold_sizes[fold_id] -= sizes[group_id]

        best_score = min(candidate_scores)
        candidates = np.flatnonzero(np.isclose(candidate_scores, best_score, rtol=0, atol=1e-15))
        selected = int(rng.choice(candidates))
        assignments[group_id] = selected
        fold_counts[selected] += counts[group_id]
        fold_sizes[selected] += sizes[group_id]

    return assignments, fold_counts, fold_sizes

fold_assignment, fold_class_counts, fold_sizes = build_group_folds(group_counts, RANDOM_SEED)
pd.DataFrame(fold_class_counts, columns=class_names).assign(total=fold_sizes)

,Center,Donut,Edge-Loc,Edge-Ring,Loc,Near-full,Random,Scratch,none,total
0,429,56,519,968,359,15,86,119,14745,17296
1,430,55,519,968,359,15,86,119,14745,17296
2,429,55,518,968,359,15,87,119,14745,17295
3,429,55,519,968,360,15,86,120,14742,17294
4,429,56,519,968,359,15,86,119,14743,17294
5,430,56,519,968,359,14,87,120,14741,17294
6,429,55,519,968,359,15,87,119,14745,17296
7,430,56,519,968,360,15,87,119,14741,17295
8,430,55,519,968,360,15,87,119,14742,17295
9,429,56,519,968,359,15,87,120,14742,17295


## Select the 7/2/1 fold combination

All possible choices of one test fold and two validation folds are compared. The objective gives equal relative importance to every class, preventing the dominant `none` class from hiding poor allocations of rare failures. Sample-size deviation is included as a secondary term.

In [5]:
target_fractions = np.array(list(TARGET_FRACTIONS.values()))
target_class_counts = target_fractions[:, None] * class_totals
target_sizes = target_fractions * group_sizes.sum()

def combination_score(class_counts, sample_counts):
    class_error = (class_counts - target_class_counts) / np.maximum(target_class_counts, 1)
    size_error = (sample_counts - target_sizes) / target_sizes
    return float(np.mean(class_error ** 2) + 0.20 * np.mean(size_error ** 2))

best = None
all_folds = set(range(10))
for test_fold in range(10):
    remaining = sorted(all_folds - {test_fold})
    for validation_folds in combinations(remaining, 2):
        train_folds = sorted(all_folds - {test_fold, *validation_folds})
        fold_sets = (train_folds, list(validation_folds), [test_fold])
        candidate_counts = np.stack([fold_class_counts[folds].sum(axis=0) for folds in fold_sets])
        candidate_sizes = np.array([fold_sizes[folds].sum() for folds in fold_sets])
        score = combination_score(candidate_counts, candidate_sizes)
        if best is None or score < best[0]:
            best = (score, fold_sets, candidate_counts, candidate_sizes)

split_score, selected_folds, split_class_counts_array, split_sizes = best
fold_to_split = {}
for split_name, folds in zip(SPLIT_NAMES, selected_folds):
    fold_to_split.update({fold: split_name for fold in folds})

lot_split = pd.Series(
    [fold_to_split[int(fold)] for fold in fold_assignment],
    index=lot_class_counts.index,
    name="split",
)
labeled["split"] = labeled["lotName"].map(lot_split)

print(f"Selected folds: {dict(zip(SPLIT_NAMES, selected_folds))}")
print(f"Objective score: {split_score:.8f}")
display(pd.Series(split_sizes, index=SPLIT_NAMES, name="samples").to_frame().assign(percentage=split_sizes / split_sizes.sum() * 100))

Selected folds: {'train': [0, 2, 3, 4, 5, 7, 8], 'validation': [1, 9], 'test': [6]}
Objective score: 0.00000799


,samples,percentage
train,121063,69.998844
validation,34591,20.000578
test,17296,10.000578


## Validate class balance and group isolation

The following tables report absolute counts and the fraction of each original class assigned to every subset. Exact percentages are not required, but every class must be present and close to its target while lot separation remains absolute.

In [6]:
class_counts = pd.crosstab(labeled["split"], labeled["failureType"]).reindex(SPLIT_NAMES)
class_allocation = class_counts.div(class_counts.sum(axis=0), axis=1)
display(class_counts)
display((class_allocation * 100).round(3))

split_indices = {name: labeled.loc[labeled["split"] == name, "row_index"].to_numpy(np.int64) for name in SPLIT_NAMES}
split_lots = {name: set(labeled.loc[labeled["split"] == name, "lotName"]) for name in SPLIT_NAMES}

assert set(split_indices["train"]).isdisjoint(split_indices["validation"])
assert set(split_indices["train"]).isdisjoint(split_indices["test"])
assert set(split_indices["validation"]).isdisjoint(split_indices["test"])
assert sum(map(len, split_indices.values())) == len(labeled)
assert set(np.concatenate(list(split_indices.values()))) == set(labeled["row_index"])
assert split_lots["train"].isdisjoint(split_lots["validation"])
assert split_lots["train"].isdisjoint(split_lots["test"])
assert split_lots["validation"].isdisjoint(split_lots["test"])
assert (class_counts > 0).all().all()

repeat_assignment, repeat_counts, repeat_sizes = build_group_folds(group_counts, RANDOM_SEED)
assert np.array_equal(fold_assignment, repeat_assignment)
assert np.array_equal(fold_class_counts, repeat_counts)
assert np.array_equal(fold_sizes, repeat_sizes)
print("All supervised disjunction, completeness, class-coverage, group-isolation, and determinism checks passed.")

failureType,Center,Donut,Edge-Loc,Edge-Ring,Loc,Near-full,Random,Scratch,none
split,,,,,,,,,
train,3006,389,3632,6776,2516,104,606,835,103199
validation,859,111,1038,1936,718,30,173,239,29487
test,429,55,519,968,359,15,87,119,14745


failureType,Center,Donut,Edge-Loc,Edge-Ring,Loc,Near-full,Random,Scratch,none
split,,,,,,,,,
train,70.005,70.09,69.994,70.0,70.025,69.799,69.977,69.992,69.998
validation,20.005,20.00,20.004,20.0,19.983,20.134,19.977,20.034,20.001
test,9.991,9.91,10.002,10.0,9.992,10.067,10.046,9.975,10.001


All supervised disjunction, completeness, class-coverage, group-isolation, and determinism checks passed.


## Assign roles to unlabeled observations

Unlabeled wafers do not participate in supervised splitting or metrics. Those from labeled training lots and from entirely unlabeled lots are eligible for later training-only methods. Unlabeled wafers associated with validation or test lots are retained but explicitly excluded from training to prevent group leakage.

In [ ]:
labeled_lots = set(lot_split.index)
unlabeled["role"] = "unlabeled_only_lot_train_eligible"
unlabeled.loc[unlabeled["lotName"].isin(split_lots["train"]), "role"] = "train_lot_train_eligible"
unlabeled.loc[unlabeled["lotName"].isin(split_lots["validation"]), "role"] = "validation_lot_excluded"
unlabeled.loc[unlabeled["lotName"].isin(split_lots["test"]), "role"] = "test_lot_excluded"

unlabeled_roles = unlabeled["role"].value_counts().rename_axis("role").rename("samples")
display(unlabeled_roles.to_frame())

eligible_mask = unlabeled["role"].isin(["train_lot_train_eligible", "unlabeled_only_lot_train_eligible"])
excluded_mask = ~eligible_mask
unlabeled_indices = unlabeled["row_index"].to_numpy(np.int64)
unlabeled_eligible_indices = unlabeled.loc[eligible_mask, "row_index"].to_numpy(np.int64)
unlabeled_excluded_indices = unlabeled.loc[excluded_mask, "row_index"].to_numpy(np.int64)

assert len(unlabeled_indices) == 638_507
assert set(unlabeled_eligible_indices).isdisjoint(unlabeled_excluded_indices)
assert set(unlabeled_eligible_indices) | set(unlabeled_excluded_indices) == set(unlabeled_indices)
assert not unlabeled.loc[eligible_mask, "lotName"].isin(split_lots["validation"] | split_lots["test"]).any()
print("Unlabeled role and leakage checks passed.")

## Save reusable split artifacts

Only row indices and compact summaries are saved; the large wafer-map arrays are not duplicated. Downstream code can reconstruct any subset with `dataset.loc[np.load(path)]`. The complete `unlabeled_indices.npy` file retains every unlabeled observation, while the eligible and excluded files enforce the training policy.

In [ ]:
for name, indices in split_indices.items():
    np.save(SPLITS_DIR / f"{name}_indices.npy", indices)

np.save(SPLITS_DIR / "unlabeled_indices.npy", unlabeled_indices)
np.save(SPLITS_DIR / "unlabeled_train_eligible_indices.npy", unlabeled_eligible_indices)
np.save(SPLITS_DIR / "unlabeled_excluded_indices.npy", unlabeled_excluded_indices)

labeled_manifest = labeled[["row_index", "lotName", "failureType", "nativeSplit", "split"]].copy()
labeled_manifest.to_csv(SPLITS_DIR / "labeled_split_manifest.csv", index=False)
class_counts.to_csv(SPLITS_DIR / "class_counts.csv")
unlabeled_roles.to_csv(SPLITS_DIR / "unlabeled_role_counts.csv")

config = {
    "random_seed": RANDOM_SEED,
    "target_fractions": TARGET_FRACTIONS,
    "group_column": "lotName",
    "target_column": "failureType",
    "selected_folds": {name: list(map(int, folds)) for name, folds in zip(SPLIT_NAMES, selected_folds)},
    "objective_score": split_score,
}
with (SPLITS_DIR / "split_config.json").open("w", encoding="utf-8") as handle:
    json.dump(config, handle, indent=2)

saved = sorted(path.name for path in SPLITS_DIR.iterdir())
print("Saved artifacts:")
print("\n".join(f"- {name}" for name in saved))

## Final interpretation

The resulting indices are the single reusable partition for all supervised pipelines. `lotName` is used only to prevent leakage and must not be passed to the classifier. The test indices are frozen. Future imbalance handling and augmentation operate only on `train_indices.npy`; future SSL or pseudo-labeling may use only `unlabeled_train_eligible_indices.npy`.